# Single-Field Continuum Imaging with AstroVIPER

This notebook is the notebook-facing counterpart of
`image_continuum_single_field.py`.

The distributed application constructs the current continuum graph:

```text
frequency-parallel continuum map
    → Taylor-product reduction
    → one global continuum model update
```

The notebook is intentionally limited to data acquisition, parameter selection,
calling the distributed application, and inspecting the result.

## 1. Start a Dask client

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## 2. Download the example Processing Set

This retains the temporary Google Drive download logic from the supplied
prototype.

In [ ]:
import glob
import os
import shutil
import zipfile

PS_STORE = "twhya_selfcal_lsrk_5chans.ps.zarr"
PS_STORE_DRIVE_ID = "1BRe3cD6YAWkn-jSPbClGGM9VlbxHP_yn"


def download_zarr_from_drive(zarr_name, file_id):
    """Download and extract one zipped Zarr directory from Google Drive."""
    if os.path.isdir(zarr_name):
        return

    import gdown

    zip_path = zarr_name + ".zip"
    gdown.download(id=file_id, output=zip_path, quiet=False)

    work_dir = zarr_name + ".extract"
    shutil.rmtree(work_dir, ignore_errors=True)
    os.makedirs(work_dir)
    shutil.move(zip_path, os.path.join(work_dir, os.path.basename(zip_path)))

    for _ in range(6):
        for root, dirs, _files in os.walk(work_dir):
            if zarr_name in dirs:
                shutil.move(os.path.join(root, zarr_name), zarr_name)
                shutil.rmtree(work_dir, ignore_errors=True)
                return

        nested_zips = glob.glob(
            os.path.join(work_dir, "**", "*.zip"),
            recursive=True,
        )
        if not nested_zips:
            break

        for nested in nested_zips:
            with zipfile.ZipFile(nested) as zf:
                zf.extractall(os.path.dirname(nested))
            os.remove(nested)

    shutil.rmtree(work_dir, ignore_errors=True)
    raise RuntimeError(f"Could not extract {zarr_name!r} from its archive.")


download_zarr_from_drive(PS_STORE, PS_STORE_DRIVE_ID)

## 3. Open and inspect the Processing Set

In [ ]:
from xradio.measurement_set import open_processing_set

ps_xdt = open_processing_set(PS_STORE)
ps_xdt.xr_ps.summary()

## 4. Define continuum imaging parameters

In [ ]:
import numpy as np

combined_field_xds = ps_xdt.xr_ps.get_combined_field_and_source_xds()
center_field_name = combined_field_xds.attrs["center_field_name"]
phase_direction = combined_field_xds.FIELD_PHASE_CENTER_DIRECTION.sel(
    field_name=center_field_name
)

image_params = {
    "image_size": [250, 250],
    "cell_size": np.array([-0.1, 0.1]) * np.pi / (180 * 3600),
    "phase_direction": phase_direction.values,
    "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
    "polarization_coords": ["I", "Q"],
    "time_coords": [0],
    "fft_padding": 1.2,
    "cpp_gridder": True,
    "nterms": 2,
    "reference_frequency_hz": 372763801257.61084,
}

imaging_weights_params = {
    "weighting": "briggs",
    "robust": 0.5,
    "casa_weighting_implementation": True,
}

iteration_control_params = {
    "niter": 100,
    "nmajor": 2,
    "threshold": 0.0,
    "gain": 0.1,
    "cyclefactor": 1.5,
    "cycleniter": -1,
    "minpsffraction": 0.05,
    "maxpsffraction": 0.8,
}

IMAGE_STORE = "twhya_continuum.img.zarr"

## 5. Run the distributed continuum application

In a repository installation, expose the function through
`astroviper.distributed_applications.imaging.__init__`. While developing the
new file directly, import it from its module path.

In [ ]:
# Repository-style import after adding the function to the package:
# from astroviper.distributed_applications.imaging import image_continuum_single_field

# Direct import when this generated module is in the current working directory:
from astroviper.distributed_applications.imaging import image_continuum_single_field

return_dict = image_continuum_single_field(
    ps_store=PS_STORE,
    image_store=IMAGE_STORE,
    image_params=image_params,
    imaging_weights_params=imaging_weights_params,
    iteration_control_params=iteration_control_params,
    gridder="prolate_spheroidal",
    deconvolver="hogbom",
    scan_intents="OBSERVE_TARGET#ON_SOURCE",
    image_data_variables_keep=[
        "sky_residual",
        "point_spread_function",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ],
    processing_set_data_group_name="base",
    single_precision_image=False,
    processing_function_threads=1,
    n_chunks=5,
    overwrite=True,
    memory_mode="in_memory",
    vizualize_graph=True,
    compute_backend="dask",
    reduce_mode="tree",
    reduce_n_batch=2,
)

## 6. Inspect timing and deconvolution output

In [ ]:
return_dict.keys(), return_dict["timing_node_tasks"]

In [ ]:
return_dict["deconvolution"]

## 7. Inspect the in-memory continuum image

Unlike the old final plotting block in the flattened script, this uses the
already computed `return_dict`; it does **not** call `dask.compute` a second
time.

In [ ]:
continuum_xds = return_dict["image"]

print(continuum_xds)
print("\nVariable dimensions:")
for variable_name, variable in continuum_xds.data_vars.items():
    print(f"{variable_name}: {variable.dims}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 9), constrained_layout=True)

r0 = continuum_xds["SKY_RESIDUAL"].isel(time=0, taylor_term=0, polarization=0)
im = axes[0, 0].imshow(r0.values, origin="lower", cmap="viridis")
axes[0, 0].set_title("Residual R0")
fig.colorbar(im, ax=axes[0, 0])

r1 = continuum_xds["SKY_RESIDUAL"].isel(time=0, taylor_term=1, polarization=0)
im = axes[0, 1].imshow(r1.values, origin="lower", cmap="viridis")
axes[0, 1].set_title("Residual R1")
fig.colorbar(im, ax=axes[0, 1])

h0 = continuum_xds["POINT_SPREAD_FUNCTION"].isel(
    time=0, psf_taylor_order=0, polarization=0
)
im = axes[1, 0].imshow(h0.values, origin="lower", cmap="viridis")
axes[1, 0].set_title("PSF H0")
fig.colorbar(im, ax=axes[1, 0])

primary_beam = continuum_xds["PRIMARY_BEAM"]
selection = {"time": 0, "polarization": 0}
if "frequency" in primary_beam.dims:
    selection["frequency"] = 0

pb = primary_beam.isel(**selection)
im = axes[1, 1].imshow(pb.values, origin="lower", cmap="viridis")
axes[1, 1].set_title("Primary Beam")
fig.colorbar(im, ax=axes[1, 1])

plt.savefig(r"./continuum.png", bbox_inches="tight")

plt.show()

## 8. Close the client

In [ ]:
viper_client.close()

In [ ]:
pb.values